# 1.0 Import Data

In [ ]:
# Import Library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

# Scikit-learn imports
from sklearn.preprocessing import MinMaxScaler, minmax_scale
from sklearn.metrics import classification_report

# Additional utilities
import random

# Check GPU availability
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [ ]:
train_df = pd.read_csv('keypoint_train_data.csv')
valid_df = pd.read_csv('keypoint_valid_data.csv')
test_df = pd.read_csv('keypoint_test_data.csv')

# 2.0 Data Preprocessing

Helper Function, class

In [ ]:
def set_seed_strict(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        
        torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed_strict(42)

In [ ]:
class SittingPosture(Dataset):
    def __init__(self, data, label):
      self.X = torch.tensor(data, dtype=torch.float32)
      self.y = torch.tensor(label, dtype=torch.long)

    def __len__(self):
      return len(self.X)

    def __getitem__(self, idx):
      X, y = self.X[idx], self.y[idx]
      return X, y

#### Data Preprocessing

In [ ]:
def minmax_normalise(df, scaler=None, is_train: bool=True):

    df_normalized = df.copy()
    
    coord_cols = df_normalized.columns.drop(['image_id', 'cat_id'])

    ####################################
    # Generated from Gemini
    ####################################
    visibility_cols = [col for col in coord_cols if col.endswith('_v')]
    xy_cols = [col for col in coord_cols if not col.endswith('_v')]
    ####################################
    # End
    ####################################

    for visibility_col in visibility_cols:
        col_name = visibility_col.replace('_v', '')
        X_col = f"{col_name}_x"
        y_col = f"{col_name}_y"
        
        # Mask here contains boolean of each row whether if visibility_col[i] is 0.0
        mask = df_normalized[visibility_col] == 0.0
        df_normalized.loc[mask, [X_col, y_col]] = np.nan

    if is_train:
        scaler = MinMaxScaler()
        df_normalized[xy_cols] = scaler.fit_transform(df_normalized[xy_cols])
        
    else:
        if scaler is None:
            print("No scaler given!")
        
        df_normalized[xy_cols] = scaler.transform(df_normalized[xy_cols])
    
    df_normalized[xy_cols] = df_normalized[xy_cols].fillna(0.0)

    if is_train:
        return df_normalized, scaler
    else:
        return df_normalized

In [ ]:
def manual_normalisation(df):
    """
    Wraps the PyTorch normalize_coco_posture function to work on a Pandas DataFrame.
    """
    df_normalized = df.copy()
    coord_cols = df_normalized.columns.drop(['image_id', 'cat_id'])

    # Iterate through the DataFrame row by row
    for index, row in df_normalized.iterrows():
        # 1. Extract the 51 flat coordinates as a numpy array
        raw_coords = row[coord_cols].values.astype(np.float32)
        
        # 2. Convert to Tensor and reshape to [17 nodes, 3 features (X,Y,V)]
        tensor_coords = torch.tensor(raw_coords).view(17, 3)
        
        # 3. Apply your custom geometric normalizer
        normalized_tensor = normalize_coco_posture_safe(tensor_coords)
        
        # 4. Flatten the [17, 3] tensor back into a flat 51-element 1D array
        flat_normalized_coords = normalized_tensor.view(-1).numpy()
        
        # 5. Inject the newly normalized coordinates back into the DataFrame
        df_normalized.loc[index, coord_cols] = flat_normalized_coords
        
    return df_normalized

def normalize_coco_posture_safe(pos_tensor):
    """
    Safely normalizes a [17, 3] COCO keypoint tensor using the Visibility flag.
    """
    # 1. Split the tensor into Coordinates and Visibility
    coords = pos_tensor[:, :2].clone() # [17, 2]
    vis = pos_tensor[:, 2].clone()     # [17]
    
    # Create a Boolean Mask based explicitly on the new visibility flag
    valid_mask = vis > 0.0 
    
    # If the skeleton is entirely missing (all visibility 0), return as-is
    if not valid_mask.any():
        return pos_tensor

    # 2. Safe Mid-Hip Calculation with Fallbacks
    l_hip_valid = valid_mask[11].item()
    r_hip_valid = valid_mask[12].item()
    
    if l_hip_valid and r_hip_valid:
        root = (coords[11] + coords[12]) / 2.0
    elif l_hip_valid:
        root = coords[11]  # Fallback to Left Hip only
    elif r_hip_valid:
        root = coords[12]  # Fallback to Right Hip only
    else:
        # Emergency Fallback: If both hips are missing, try Shoulders (5, 6)
        l_sho_valid = valid_mask[5].item()
        r_sho_valid = valid_mask[6].item()
        if l_sho_valid and r_sho_valid:
            root = (coords[5] + coords[6]) / 2.0
        else:
            root = torch.tensor([0.0, 0.0], device=coords.device)

    # 3. Apply Centering ONLY to valid points
    # The missing 0.0 points remain safely dead at the origin
    coords[valid_mask] = coords[valid_mask] - root
    
    # 4. Aspect-Preserving Scale ONLY on valid points
    min_vals = coords[valid_mask].min(dim=0)[0]
    max_vals = coords[valid_mask].max(dim=0)[0]
    ranges = max_vals - min_vals
    global_scale = ranges.max()
    
    # Scale valid points. Add epsilon to prevent division by zero
    coords[valid_mask] = coords[valid_mask] / (global_scale + 1e-6)
    
    # 5. Recombine the normalized coordinates with the untouched visibility column
    # We use unsqueeze(1) to turn vis back from [17] to [17, 1] for concatenation
    final_tensor = torch.cat([coords, vis.unsqueeze(1)], dim=1)
    
    return final_tensor

In [ ]:
def normalise(df, is_train:bool, type: str=None, scaler=None, variation: str=None):
    if type == "manual":
        df_normalized = manual_normalisation(df)
        return df_normalized
        
    elif type == "auto" and variation in ["row_wise", "col_wise"]:
        
        if is_train and variation == "col_wise":
            df_normalized, scaler = minmax_normalise(df=df, variations=variation, scaler=scaler, is_train=is_train)
            return df_normalized, scaler
            
        else:
            df_normalized = minmax_normalise(df=df, variations=variation, scaler=scaler, is_train=is_train)
            return df_normalized
            
    else:
        print("Unknown type or variation given.")
        raise ValueError("Normalization failed due to invalid arguments.")

In [ ]:
#1st Norm
train_df_norm = manual_normalisation(train_df)
train_df_norm = train_df_norm.drop(['image_id', 'cat_id'],axis=1)
y_train = train_df['cat_id']
validate_df_norm = manual_normalisation(valid_df)
validate_df_norm = validate_df_norm.drop(['image_id', 'cat_id'],axis=1)
y_valid = valid_df['cat_id']
test_df_norm = manual_normalisation(test_df)
test_df_norm = test_df_norm.drop(['image_id', 'cat_id'],axis=1)
y_test = test_df['cat_id']

In [ ]:
"""
#2nd Norm
train_df_norm, fitted_scaler = normalise(df=train_df, is_train=True, type="auto")
validate_df_norm = normalise(df=valid_df, is_train=False, type="auto", scaler=fitted_scaler)
test_df_norm = normalise(df=test_df, is_train=False, type="auto", scaler=fitted_scaler)

train_df_norm = train_df_norm.drop(['image_id', 'cat_id'],axis=1)
validate_df_norm = validate_df_norm.drop(['image_id', 'cat_id'],axis=1)
test_df_norm = test_df_norm.drop(['image_id', 'cat_id'],axis=1)

y_train = train_df['cat_id']
y_valid = valid_df['cat_id']
y_test = test_df['cat_id']
"""

# 3.0 Building Models

#### Create Datasets

In [ ]:
# Change labels from (1, 2) to (0, 1)
y_train = y_train.values - 1
y_valid = y_valid.values - 1
y_test = y_test.values - 1

In [ ]:
train_dataset = SittingPosture(train_df_norm.values, y_train)
valid_dataset = SittingPosture(validate_df_norm.values, y_valid)
test_dataset = SittingPosture(test_df_norm.values, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader =  DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader =  DataLoader(test_dataset, batch_size=64, shuffle=False)

Mini Gradient Descent

In [ ]:
# Get a batch of data
x_batch, y_batch = next(iter(train_loader))
print(f"Shape of x_batch: {x_batch.shape}")
print(f"Shape of y_batch: {y_batch.shape}")
# Get a batch of data
x_valid_batch, y_valid_batch = next(iter(valid_loader))
print(f"Shape of x_batch: {x_batch.shape}")
print(f"Shape of y_batch: {y_batch.shape}")
# Get a batch of data
x_test_batch, y_test_batch = next(iter(test_loader))
print(f"Shape of x_batch: {x_batch.shape}")
print(f"Shape of y_batch: {y_batch.shape}")

#### Fully Connected Neuron Network

##### Create FCNN

In [ ]:
#Add Dropout???
class FCNN(nn.Module):
  def __init__(self):
    """
    Create a Fully Connected Neuron Network

    """
    super().__init__()
    self.layer1 = nn.Sequential(
        nn.Linear(in_features=51, out_features=128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.5),
    )
    self.layer2 = nn.Sequential(
        nn.Linear(in_features=128, out_features=64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.2)
    )
    self.layer3 = nn.Sequential(
        nn.Linear(in_features=64, out_features=32),
        nn.BatchNorm1d(32),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.2)
    )
    self.outputlayer = nn.Linear(in_features=32, out_features=1)

  def forward(self, x):
    """
    To perform inference (do prediction).
    During training mode, this will construct the computational graph
    """
    # perform forward propagation and build computational graph
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.outputlayer(x)

    return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fcnn = FCNN()
fcnn.to(device)
print(fcnn)

##### Cost Function and Optimizer

In [ ]:
optimizer = optim.AdamW(fcnn.parameters(), lr=0.0001, weight_decay= 1e-3)
loss_function = nn.BCEWithLogitsLoss()

Scheduler

In [ ]:
warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=5)

decay_scheduler = CosineAnnealingLR(optimizer, T_max=(200 - 5), eta_min=1e-6)

scheduler = SequentialLR(
    optimizer, 
    schedulers=[warmup_scheduler, decay_scheduler], 
    milestones=[5]
)

# 4.0 Training the Model

Helper Function

In [ ]:
def train(train_loader, model, optimizer, loss_function, num_epochs, patience):
    # List to store epoch costs for plotting
    train_epoch_costs = []
    valid_epoch_costs = []
    train_epoch_accuracy = []
    valid_epoch_accuracy = []
    patience_count = 0
    best_val_loss = None

    # For all epochs
    for epoch in range(num_epochs):
        train_total = 0
        train_correct = 0
        train_batch_costs = []
        valid_batch_costs = []

        # set to training mode
        model.train()

        # ===== TRAIN ======
        for x_batch, y_batch in train_loader:

            # transfer batch data to the GPU if available
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            # Reshape y_batch's shape = y_hat's shape (B, 1)
            y_batch = y_batch.unsqueeze(1).float()

            # Forward propagation
            y_hat = model(x_batch) # shape = (B,1)

            # Compute cost
            cost = loss_function(y_hat, y_batch)
            train_batch_costs.append(cost.item())

            # Compute accuracy
            y_hat = y_hat.squeeze()
            y_batch = y_batch.squeeze()
            y_pred = (y_hat > 0.5).float()
            train_total   += y_batch.size(0)
            train_correct += (y_pred == y_batch).sum().item()

            # Backward propagation
            cost.backward()

            # Update parameters
            optimizer.step()

            # Reset the gradient
            optimizer.zero_grad()
            
        # ===== END OF TRAIN ======

        # ===== Scheduler =====
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        train_acc = train_correct / train_total

        total = 0
        correct = 0
        # set to evaluation mode
        model.eval()

        # ===== VALIDATION ======
        with torch.inference_mode():
          for x_batch, y_batch in valid_loader:
            
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            y_batch = y_batch.unsqueeze(1).float()

            y_hat = model(x_batch) # shape = (B,1)

            # Compute cost
            cost = loss_function(y_hat, y_batch)
            valid_batch_costs.append(cost.item())

            # Compute accuracy
            y_pred = (y_hat.squeeze() > 0.5).float()
            y_batch = y_batch.squeeze()
            total   += y_batch.size(0)
            correct += (y_pred == y_batch).sum().item()

        val_acc = correct / total
        # ===== END OF VALIDATION ======

        # Calculate average cost/avg for the epoch
        avg_train_epoch_cost = sum(train_batch_costs) / len(train_batch_costs)
        avg_valid_epoch_cost = sum(valid_batch_costs) / len(valid_batch_costs)
        train_epoch_costs.append(avg_train_epoch_cost)
        valid_epoch_costs.append(avg_valid_epoch_cost)
        train_epoch_accuracy.append(train_acc)
        valid_epoch_accuracy.append(val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}], LR: {current_lr:.6f}, average train cost: {avg_train_epoch_cost:.4f}, average val cost: {avg_valid_epoch_cost:.4f}, average train accuracy: {train_acc:.4f}, average val accuracy: {val_acc:.4f}")

        if best_val_loss is None or avg_valid_epoch_cost < best_val_loss:
          best_val_loss = avg_valid_epoch_cost
          torch.save(model.state_dict(), 'best_model.pth')
          patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
              break
    # Plotting the cost
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(train_epoch_costs, label='Training Loss')
    ax1.plot(valid_epoch_costs, label='Validation Loss', linestyle='--')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(train_epoch_accuracy, label='Training Accuracy')
    ax2.plot(valid_epoch_accuracy, label='Validation Accuracy', linestyle='--')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.show()

#### Fully Connected Neuron Network

In [ ]:
train(train_loader=train_loader, model=fcnn, optimizer=optimizer, loss_function=loss_function, num_epochs=200, patience=30)

# 5.0 Model Evaluation

#### Fully Connected Neuron Network

In [ ]:
def evaluate(data_loader, net):
    correct = 0
    total = 0

    # Set the model to evaluation mode
    net.eval()

    # Disable gradient computation
    with torch.inference_mode():

        # Repeat for all batch data
         for x_batch, y_batch in data_loader:

            # transfer to GPU
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            # predict class scores
            y_hat = net(x_batch)
            y_hat = y_hat.squeeze()
            y_pred = (y_hat > 0.5).float()

            # accumulate to total and correct
            total   += y_batch.size(0)
            correct += (y_pred == y_batch).sum().item()

    # compute accuracy
    acc = correct / total
    return acc

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

In [ ]:
test_acc = evaluate(valid_loader, model)
print(f"t_acc = {test_acc * 100:.2f}%")

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in valid_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.5).squeeze().cpu().numpy()
        all_preds.extend(predicted)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))


In [ ]:
test_acc = evaluate(test_loader, fcnn)
print(f"t_acc = {test_acc * 100:.2f}%")

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.5).squeeze().cpu().numpy()
        all_preds.extend(predicted)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))
